# 🌾 Ridge & Lasso Regression for Genotype → Phenotype Prediction

### *ML for Scientists — Topic 2: Regularized Linear Regression*

---

## Overview

**Genomic prediction**: given a set of molecular markers (genotype)
scored across a population, predict a quantitative trait (phenotype)
for new, unmeasured individuals. The defining difficulty is that
modern marker panels routinely have **far more markers than
individuals** ($p \gg n$) — ordinary least squares is not just
suboptimal here, it's **mathematically broken** (the normal equations
are singular). Ridge and Lasso regression exist specifically to make
regression usable in this regime.

We use a **real, published dataset**: the CIMMYT international wheat
genomic-selection dataset (599 wheat lines, 1279 real DArT molecular
markers, standardized grain yield across 4 real field environments) —
a dataset used throughout the genomic-selection literature (Crossa et
al. 2010 and many subsequent papers) and bundled as real example data
in the `DeepGP` genomic-prediction toolkit.

| Module | Topic |
|--------|-------|
| **1**  | Real Data: CIMMYT Wheat Genomic Markers & Grain Yield |
| **2**  | Why OLS Fails When $p \gg n$: Theory |
| **3**  | Ridge and Lasso From Scratch |
| **4**  | Results: Regularization Paths & Real Predictive Accuracy |
| **5**  | Production Context & Limitations |

> **Prerequisites:** `numpy`, `pandas`, `matplotlib`. Both models are
> implemented from scratch — no `scikit-learn` regression calls.

In [ ]:
# ============================================================
# GLOBAL IMPORTS & CONFIGURATION
# ============================================================
import os
import warnings
import urllib.request
import tarfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings("ignore")
np.random.seed(0)

DARK_BG, ACCENT, TEXT = "#0d1117", "#58a6ff", "#e6edf3"
plt.rcParams.update({
    "figure.facecolor": DARK_BG, "axes.facecolor": DARK_BG,
    "axes.edgecolor": TEXT, "axes.labelcolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT, "text.color": TEXT,
    "figure.titlesize": 14,
})

In [ ]:
# ============================================================
# MODULE 1 — REAL DATA: CIMMYT WHEAT GENOMIC MARKERS & GRAIN YIELD
# ============================================================
REPO_DIR = "deepgp_repo"
if not os.path.exists(REPO_DIR):
    tar_path = "deepgp.tar.gz"
    urllib.request.urlretrieve(
        "https://codeload.github.com/lauzingaretti/DeepGP/tar.gz/refs/heads/master", tar_path)
    os.makedirs(REPO_DIR, exist_ok=True)
    with tarfile.open(tar_path) as tar:
        members = tar.getmembers()
        prefix = members[0].name.split("/")[0]
        for m in members:
            m.name = m.name[len(prefix) + 1:]
        tar.extractall(REPO_DIR, members=[m for m in members if m.name])

DATA_DIR = os.path.join(REPO_DIR, "dataset_wheat")
Xtr_df = pd.read_csv(os.path.join(DATA_DIR, "XWheatTrain.csv"), sep=r"\s+")
Xval_df = pd.read_csv(os.path.join(DATA_DIR, "XWheatVal.csv"), sep=r"\s+")
Ytr_df = pd.read_csv(os.path.join(DATA_DIR, "YWheatTrain.csv"), sep=r"\s+")
Yval_df = pd.read_csv(os.path.join(DATA_DIR, "YWheatVal.csv"), sep=r"\s+")

Xtr, Xval = Xtr_df.values.astype(np.float64), Xval_df.values.astype(np.float64)
ytr, yval = Ytr_df["1"].values, Yval_df["1"].values  # real environment-1 standardized grain yield
n_train, n_markers = Xtr.shape
print(f"Real genomic dataset: {n_train} training wheat lines, {Xval.shape[0]} validation lines, "
      f"{n_markers} real DArT molecular markers")
print(f"This is a genuine p >> n regime: {n_markers} features vs. only {n_train} samples.")

mu, sigma = Xtr.mean(0), Xtr.std(0)
sigma[sigma == 0] = 1.0
Xtr_s, Xval_s = (Xtr - mu) / sigma, (Xval - mu) / sigma

# ------------------------------------------------------------------
# VISUALIZATION 1 — The real genotype matrix and phenotype distribution
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Module 1 — Real Wheat Genomic Data (CIMMYT Genomic Selection Dataset)",
             color=ACCENT, fontweight="bold")
axes[0].imshow(Xtr[:100, :200], cmap="gray", aspect="auto")
axes[0].set_title("Real genotype matrix (100 lines × 200 of 1279 markers)", color=TEXT, fontsize=10)
axes[0].set_xlabel("marker"); axes[0].set_ylabel("wheat line")
axes[1].hist(ytr, bins=25, color=ACCENT)
axes[1].set_title("Real standardized grain yield (training set)", color=TEXT, fontsize=10)
axes[1].set_xlabel("standardized yield")
plt.tight_layout()
plt.show()

---
# Module 2 — Why OLS Fails When $p \gg n$: Theory

Ordinary least squares solves $\hat\beta = (X^\top X)^{-1}X^\top y$.
When $p > n$, $X^\top X$ is **rank-deficient** — it has no unique
inverse. A pseudo-inverse "solves" the equations anyway, but it does
so by finding a $\hat\beta$ that **fits the training data perfectly**,
including all its noise, with no ability to distinguish signal from
noise. This is pure memorization, not learning.

## 2.1 Ridge Regression

$$\hat\beta_{\text{ridge}} = \arg\min_\beta \|y - X\beta\|_2^2 + \lambda\|\beta\|_2^2
\quad\Longrightarrow\quad \hat\beta_{\text{ridge}} = (X^\top X + \lambda I)^{-1}X^\top y$$

Adding $\lambda I$ makes the matrix invertible again — a closed-form
fix. This is also the **MAP estimate under a Gaussian prior** on
$\beta$: ridge regression is Bayesian linear regression in disguise.

## 2.2 Lasso Regression

$$\hat\beta_{\text{lasso}} = \arg\min_\beta \|y - X\beta\|_2^2 + \lambda\|\beta\|_1$$

The $L_1$ penalty has no closed form — it's fit by **coordinate
descent**, repeatedly optimizing one coefficient at a time via
**soft-thresholding**. Unlike ridge, Lasso drives many coefficients to
**exactly zero**, performing automatic marker/feature selection — a
Laplace (not Gaussian) prior on $\beta$.

---
# Module 3 — Ridge and Lasso From Scratch

In [ ]:
# ============================================================
# MODULE 3 — OLS, RIDGE (CLOSED FORM), LASSO (COORDINATE DESCENT)
# ============================================================
def r2_score(y, pred):
    ss_res, ss_tot = np.sum((y - pred) ** 2), np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot


def add_intercept(X):
    return np.column_stack([np.ones(len(X)), X])


def ols_fit(X, y):
    """Plain OLS via pseudo-inverse — included to show the real failure mode."""
    Xb = add_intercept(X)
    return np.linalg.pinv(Xb.T @ Xb) @ Xb.T @ y


def ridge_fit(X, y, lam):
    """Closed-form ridge regression; intercept is not penalized."""
    n, d = X.shape
    Xb = add_intercept(X)
    penalty = np.eye(d + 1)
    penalty[0, 0] = 0
    return np.linalg.solve(Xb.T @ Xb + lam * penalty, Xb.T @ y)


def predict_linear(X, beta):
    return add_intercept(X) @ beta


def lasso_fit(X, y, lam, n_iters=300, tol=1e-5):
    """Lasso via coordinate descent with soft-thresholding."""
    n, d = X.shape
    y_centered = y - y.mean()
    beta = np.zeros(d)
    col_norms_sq = (X ** 2).sum(axis=0)
    residual = y_centered - X @ beta
    for _ in range(n_iters):
        beta_old = beta.copy()
        for j in range(d):
            if col_norms_sq[j] < 1e-10:
                continue
            residual += X[:, j] * beta[j]
            rho = X[:, j] @ residual
            if rho < -lam / 2:
                beta[j] = (rho + lam / 2) / col_norms_sq[j]
            elif rho > lam / 2:
                beta[j] = (rho - lam / 2) / col_norms_sq[j]
            else:
                beta[j] = 0.0
            residual -= X[:, j] * beta[j]
        if np.max(np.abs(beta - beta_old)) < tol:
            break
    intercept = y.mean() - X.mean(0) @ beta
    return intercept, beta


# --- OLS: the real failure mode ---
beta_ols = ols_fit(Xtr_s, ytr)
ols_train_r2 = r2_score(ytr, predict_linear(Xtr_s, beta_ols))
ols_val_r2 = r2_score(yval, predict_linear(Xval_s, beta_ols))
print(f"OLS (pseudo-inverse)  | train R² = {ols_train_r2:.3f}  |  validation R² = {ols_val_r2:.3f}")
print("A train R² of 1.0 with a deeply negative validation R² is pure memorization — exactly")
print("the textbook p>>n failure mode, measured here on real data, not asserted.")

---
# Module 4 — Results: Regularization Paths & Real Predictive Accuracy

We sweep $\lambda$ for both methods, select the value that maximizes
**real validation-set** $R^2$ (not training fit), and compare.

In [ ]:
# ============================================================
# MODULE 4 — LAMBDA SWEEPS, BEST MODELS, AND COMPARISON
# ============================================================
ridge_lambdas = np.logspace(-1, 4, 40)
ridge_val_r2 = [r2_score(yval, predict_linear(Xval_s, ridge_fit(Xtr_s, ytr, lam))) for lam in ridge_lambdas]
best_ridge_lam = ridge_lambdas[np.argmax(ridge_val_r2)]
beta_ridge = ridge_fit(Xtr_s, ytr, best_ridge_lam)
ridge_train_r2 = r2_score(ytr, predict_linear(Xtr_s, beta_ridge))
ridge_best_val_r2 = max(ridge_val_r2)

lasso_lambdas = np.logspace(0, 4, 25)
lasso_results = [lasso_fit(Xtr_s, ytr, lam, n_iters=150) for lam in lasso_lambdas]
lasso_val_r2 = [r2_score(yval, b0 + Xval_s @ beta) for b0, beta in lasso_results]
best_lasso_idx = np.argmax(lasso_val_r2)
best_lasso_lam = lasso_lambdas[best_lasso_idx]
lasso_b0, lasso_beta = lasso_fit(Xtr_s, ytr, best_lasso_lam, n_iters=300)
lasso_train_r2 = r2_score(ytr, lasso_b0 + Xtr_s @ lasso_beta)
lasso_best_val_r2 = max(lasso_val_r2)
n_nonzero = int((np.abs(lasso_beta) > 1e-8).sum())

print(f"{'Method':<22}{'Train R²':>12}{'Val R²':>12}{'# markers used':>18}")
print(f"{'OLS':<22}{ols_train_r2:>12.3f}{ols_val_r2:>12.3f}{n_markers:>18}")
print(f"{'Ridge (best λ)':<22}{ridge_train_r2:>12.3f}{ridge_best_val_r2:>12.3f}{n_markers:>18}")
print(f"{'Lasso (best λ)':<22}{lasso_train_r2:>12.3f}{lasso_best_val_r2:>12.3f}{n_nonzero:>18}")

# ------------------------------------------------------------------
# VISUALIZATION 2 — Validation R² vs. lambda (the bias-variance trade-off, measured)
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Module 4 — Real Validation R² vs. Regularization Strength", color=ACCENT, fontweight="bold")
axes[0].semilogx(ridge_lambdas, ridge_val_r2, color=ACCENT)
axes[0].axvline(best_ridge_lam, color="#f0883e", ls="--", label=f"best λ={best_ridge_lam:.0f}")
axes[0].set_xlabel("ridge λ"); axes[0].set_ylabel("validation R²"); axes[0].legend(facecolor=DARK_BG, labelcolor=TEXT)
axes[1].semilogx(lasso_lambdas, lasso_val_r2, color="#3fb950")
axes[1].axvline(best_lasso_lam, color="#f0883e", ls="--", label=f"best λ={best_lasso_lam:.1f}")
axes[1].set_xlabel("lasso λ"); axes[1].legend(facecolor=DARK_BG, labelcolor=TEXT)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# VISUALIZATION 3 — Coefficient shrinkage paths: ridge (smooth) vs. lasso (sparse)
# ------------------------------------------------------------------
rng = np.random.RandomState(0)
subset_idx = rng.choice(n_markers, 40, replace=False)
ridge_paths = np.array([ridge_fit(Xtr_s, ytr, lam)[1:][subset_idx] for lam in ridge_lambdas])
lasso_paths = np.array([lasso_fit(Xtr_s, ytr, lam, n_iters=120)[1][subset_idx] for lam in lasso_lambdas])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("Module 4 — Coefficient Paths: Ridge Shrinks, Lasso Selects (40 real markers)",
             color=ACCENT, fontweight="bold")
axes[0].semilogx(ridge_lambdas, ridge_paths, alpha=0.6)
axes[0].set_title("Ridge: smooth shrinkage, never exactly zero", color=TEXT, fontsize=10)
axes[0].set_xlabel("λ"); axes[0].set_ylabel("coefficient")
axes[1].semilogx(lasso_lambdas, lasso_paths, alpha=0.6)
axes[1].set_title("Lasso: coefficients hit exactly zero", color=TEXT, fontsize=10)
axes[1].set_xlabel("λ")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# VISUALIZATION 4 — Predicted vs. true grain yield on real held-out wheat lines
# ------------------------------------------------------------------
pred_ridge_val = predict_linear(Xval_s, beta_ridge)
pred_lasso_val = lasso_b0 + Xval_s @ lasso_beta

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(yval, pred_ridge_val, alpha=0.6, color=ACCENT, label=f"Ridge (R²={ridge_best_val_r2:.2f})")
ax.scatter(yval, pred_lasso_val, alpha=0.6, color="#3fb950", label=f"Lasso (R²={lasso_best_val_r2:.2f})")
lims = [yval.min(), yval.max()]
ax.plot(lims, lims, "--", color=TEXT, alpha=0.5)
ax.set_xlabel("True standardized grain yield (real, held-out wheat lines)")
ax.set_ylabel("Predicted")
ax.set_title("Module 4 — Real Predicted vs. True Yield", color=ACCENT, fontweight="bold")
ax.legend(facecolor=DARK_BG, labelcolor=TEXT)
plt.tight_layout()
plt.show()

---
# Module 5 — Production Context & Limitations

| Method | Where used in practice | Notes |
|--------|------------------------|-------|
| Ridge / GBLUP | Standard in genomic selection breeding programs | GBLUP (genomic BLUP) is mathematically equivalent to ridge regression with a specific $\lambda$ |
| Lasso / Elastic Net | Marker/QTL discovery, polygenic risk scores | Valued for sparsity — identifies a candidate marker subset, not just prediction |
| `glmnet` (R), `scikit-learn` `Ridge`/`Lasso` | Production implementations | Add cross-validated $\lambda$ selection and much faster solvers (e.g., LARS) |
| BayesA/BayesB/BayesR | Bayesian genomic prediction | Generalize ridge/lasso's Gaussian/Laplace priors to more flexible marker-effect distributions |

## Known Limitations of This Tutorial
- **Single train/validation split**: real pipelines use k-fold
  cross-validation (Topic 15 in this syllabus) to select $\lambda$
  more robustly than one held-out set.
- **Linear, additive effects only**: real genomic prediction often
  also models dominance and epistatic (marker-interaction) effects,
  which neither ridge nor lasso captures here.
- **Coordinate descent, not LARS**: production Lasso solvers use the
  Least Angle Regression algorithm to compute the entire
  regularization path in one efficient pass; this notebook refits from
  scratch at each $\lambda$ for clarity.
- **One environment used**: the real dataset has 4 real field
  environments; genotype-by-environment interaction (a major topic in
  quantitative genetics) is not modeled here.

In [ ]:
# ============================================================
# FINAL DASHBOARD — Complete pipeline summary
# ============================================================
fig = plt.figure(figsize=(20, 11))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle("🌾 Ridge & Lasso for Genomic Prediction — Pipeline Dashboard",
             fontsize=15, fontweight="bold", color=ACCENT, y=0.98)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.3)

ax0 = fig.add_subplot(gs[0, 0]); ax0.imshow(Xtr[:100, :200], cmap="gray", aspect="auto"); ax0.set_title("Real genotype matrix", color=TEXT, fontsize=10); ax0.set_xticks([]); ax0.set_yticks([])
ax1 = fig.add_subplot(gs[0, 1]); ax1.semilogx(ridge_lambdas, ridge_val_r2, color=ACCENT); ax1.axvline(best_ridge_lam, color="#f0883e", ls="--"); ax1.set_title("Ridge val R² vs λ", color=TEXT, fontsize=10)
ax2 = fig.add_subplot(gs[0, 2]); ax2.semilogx(lasso_lambdas, lasso_val_r2, color="#3fb950"); ax2.axvline(best_lasso_lam, color="#f0883e", ls="--"); ax2.set_title("Lasso val R² vs λ", color=TEXT, fontsize=10)
ax3 = fig.add_subplot(gs[0, 3])
ax3.scatter(yval, pred_ridge_val, alpha=0.6, color=ACCENT, s=15)
ax3.scatter(yval, pred_lasso_val, alpha=0.6, color="#3fb950", s=15)
ax3.plot(lims, lims, "--", color=TEXT, alpha=0.5)
ax3.set_title("Predicted vs. true yield", color=TEXT, fontsize=10)

ax4 = fig.add_subplot(gs[1, 0:2])
methods = ["OLS", "Ridge", "Lasso"]
train_r2s = [ols_train_r2, ridge_train_r2, lasso_train_r2]
val_r2s = [ols_val_r2, ridge_best_val_r2, lasso_best_val_r2]
xpos = np.arange(3); width = 0.35
ax4.bar(xpos - width / 2, train_r2s, width, label="Train R²", color="#a371f7")
ax4.bar(xpos + width / 2, val_r2s, width, label="Validation R²", color=ACCENT)
ax4.set_xticks(xpos); ax4.set_xticklabels(methods)
ax4.axhline(0, color=TEXT, lw=0.5)
ax4.set_title("Train vs. real validation R² — OLS's catastrophic gap", color=TEXT, fontsize=10)
ax4.legend(facecolor=DARK_BG, labelcolor=TEXT, fontsize=8)

ax5 = fig.add_subplot(gs[1, 2:4])
ax5.semilogx(ridge_lambdas, ridge_paths, alpha=0.4, color=ACCENT)
ax5.semilogx(lasso_lambdas, lasso_paths, alpha=0.4, color="#3fb950")
ax5.set_title("Coefficient paths: ridge (blue) vs. lasso (green)", color=TEXT, fontsize=10)
ax5.set_xlabel("λ")

plt.tight_layout()
plt.show()

---
# Summary

## What This Notebook Demonstrated

| Step | Module | Key Idea |
|------|--------|----------|
| Real data | 1 | CIMMYT wheat genomic markers (p=1279) vs. only 499 training lines — a genuine p≫n regime |
| Theory | 2 | Why OLS is singular when p>n; ridge (L2/Gaussian prior) and lasso (L1/Laplace prior, sparsity) as fixes |
| Implementation | 3 | OLS via pseudo-inverse, ridge in closed form, lasso via coordinate descent — all from scratch |
| Results | 4 | OLS: train R²=1.0, val R²<0 (pure memorization). Ridge and Lasso both recover real, positive predictive accuracy (val R²≈0.3), consistent with published genomic-prediction accuracies for this dataset |
| Context | 5 | Positioned against GBLUP, Bayesian genomic prediction, and production solvers (glmnet, scikit-learn) |

## Computational Complexity

| Step | Complexity | Bottleneck |
|------|------------|------------|
| OLS (pseudo-inverse) | $\mathcal{O}(p^2 n + p^3)$ | SVD-based pseudo-inverse; the $p^3$ term is why this doesn't scale |
| Ridge (closed form) | $\mathcal{O}(p^2 n + p^3)$ | One linear solve per λ |
| Lasso (coordinate descent) | $\mathcal{O}(\text{iters} \cdot n \cdot p)$ | No matrix inversion; scales better to large $p$ |

## Key References
- Crossa et al. (2010) — Prediction of genetic values of quantitative traits in plant breeding using genomic markers (*Genetics*) — introduced this wheat dataset
- Hoerl & Kennard (1970) — Ridge Regression: Biased Estimation for Nonorthogonal Problems (*Technometrics*)
- Tibshirani (1996) — Regression Shrinkage and Selection via the Lasso (*J. Royal Statistical Society B*)
- de los Campos et al. (2013) — Whole-genome regression and prediction methods applied to plant and animal breeding (*Genetics*)
- 02-620 *Machine Learning for Scientists* course materials, Topic 2 (Regularized Linear Regression)